# Lab 10 - Dataset Engineering

**Chapter 8, "Dataset Engineering"** (data curation and quality, data synthesis and verification, deduplication, cleaning, formatting) plus the **"When to Finetune"** part of **Chapter 7** | **Environment:** cluster notebook (local Docker JupyterLab + KU VPN) | **Time:** about 4-5 hours

Next week (Lab 11) you will fine-tune a small model on Colab. (Fine-tuning means training an existing model a little more on your own examples, so it learns your task or style.) A fine-tune is only as good as its data, so this week you build that data yourself, the way an AI engineer at Larkspur would:

- **Part A (guided):** audit the Harbor PSA ticket data for quality problems. Find near-duplicate tickets with embeddings, find suspicious category and priority labels with a nearest-neighbor check plus an LLM relabeling pass, and clean the data.
- **Part B (applied):** build a supervised fine-tuning (SFT) dataset in chat format: synthetic question/answer pairs grounded in **current** Larkspur documents (generated by `qwen-moe`, filtered by an AI judge and heuristics), plus real "ticket -> resolution note" pairs from the cleaned tickets. You will save about 200-400 training examples and a 40-example test split, with **lineage** for every example.

**Before you start:** KU VPN on, `docker compose up` running, notebook opened from `labs/lab10-dataset-engineering/`. The whole lab sends about 145 requests; the long steps run 3 requests in parallel, meaning at the same time instead of one after another (your key allows 4 at once). If your dataset build fails, a pre-built fallback dataset is in `fallback_dataset/` so you can still do Lab 11.

## Words you'll meet in this lab

You don't need to memorize these. Each one is explained again where it first matters; come back here if a word stops you.

| Term | What it means | Why it matters |
|---|---|---|
| **Harbor PSA** | Larkspur's ticketing system (PSA = professional services automation: the software an IT services company uses to log tickets, time, and billing). | The ticket data you clean comes from it. |
| **Fine-tuning** | Training an existing model a little more on your own examples, so it adapts to your task or style. | Lab 11 fine-tunes a small model on the dataset you build this week. |
| **SFT (supervised fine-tuning)** | Fine-tuning on pairs of an input and the exact output you want the model to give. | Every example you save is one such pair. |
| **Training example** | One input plus its desired output (here, a short chat: question and answer). | The model learns only from these, mistakes included. |
| **Label, label noise** | A label is a value someone assigned to a record, like a ticket's category or priority. Label noise means some labels are wrong. | A model trained on wrong labels learns the wrong thing. |
| **Data profiling** | Quick counts and checks that show what's in the data and what's wrong with it. | Cheap, and it tells you where to spend expensive LLM calls. |
| **Embedding** | A list of numbers that captures the meaning of a text; texts with similar meaning get similar lists (from Lab 08). | Lets you find near-duplicate and similar tickets. |
| **Cosine similarity** | A score of how alike two embeddings are: 1.0 means the same meaning, values near 0 mean unrelated. | You flag two tickets as duplicates when their score is above a threshold. |
| **Deduplication** | Finding and removing repeated examples, exact copies or near-copies. | Duplicates make a model over-learn a few examples. |
| **Nearest neighbors** | The items most similar to a given item (here, by cosine similarity). | If a ticket's neighbors mostly have another category, its label is suspicious. |
| **Control group** | Items picked at random, not because anything looked wrong, and checked the same way. | Shows how often the checker itself disagrees with labels that are probably fine. |
| **Synthetic data** | Training data written by an AI model instead of collected from real people or events. | Fast and cheap, but it can be wrong in ways that are hard to spot. |
| **Grounded, evidence** | An answer is grounded when the source document supports it. Evidence is a phrase copied word for word from the source. | You check the evidence yourself instead of trusting the model. |
| **AI judge** | A model that scores another model's output (from Lab 04). | Filters out unfaithful pairs before they reach training. |
| **Heuristic** | A simple rule-of-thumb check, like "the answer is 20-800 characters long". | Free (no model calls) and easy to explain. |
| **Lineage** | A record of where each example came from: which document and version, or which ticket, and which models made it. | If a source turns out to be wrong, you know which examples to remove. |
| **Train/test split, held out** | The training split is what the model learns from. The test split is kept aside (held out) and used only to measure the model afterwards. | Lab 11 measures your fine-tune on the test split. |
| **Data leakage** | When test examples (or near-copies of them) end up in the training data. | The test score then measures memory, not skill. |
| **JSONL, chat format** | JSONL is a text file with one JSON object per line. Chat format stores each example as a list of messages with roles (`system`, `user`, `assistant`). | It's the file format Lab 11's training code reads. |

## Setup

The setup cell loads the tickets and contacts from `larkspur.db` into **pandas DataFrames** (tables you can filter and count in Python) and defines two helpers used all lab: `parallel_map` runs a function on many items, a few at a time, and `chat_json` asks a model for a reply that must match a JSON schema (structured output, from Lab 06), so the reply can be read as data. If a call fails, `chat_json` prints a short error in parentheses and returns `None`.

In [ ]:
import json, random, re, sqlite3, time
from collections import Counter
from concurrent.futures import ThreadPoolExecutor
from difflib import SequenceMatcher
from pathlib import Path

import numpy as np
import pandas as pd
from itec632 import get_client, MODELS, NO_THINK, strip_think
from itec632.corpus import db_path, load_docs
from itec632.grading import LabChecker

client = get_client()
ck = LabChecker("lab10", part="cluster")
random.seed(632)
GEN_MODEL, JUDGE_MODEL = MODELS["main"], MODELS["gemma"]   # judge from a different family (Ch. 3: self-bias)
OUT = Path("my_dataset"); OUT.mkdir(exist_ok=True)

available = {m.id for m in client.models.list().data}
print({m: (m in available) for m in [GEN_MODEL, JUDGE_MODEL, MODELS["embed"]]})

con = sqlite3.connect(f"file:{db_path()}?mode=ro", uri=True)
tickets = pd.read_sql("SELECT t.*, c.tier FROM tickets t JOIN clients c ON c.code = t.client_code", con)
contacts = pd.read_sql("SELECT * FROM contacts", con)
tickets["created"] = pd.to_datetime(tickets["created_at"])
print(len(tickets), "tickets,", len(contacts), "contacts")


def parallel_map(fn, items, workers=3):
    """Run fn over items with a few parallel requests (the gateway allows 4 per key)."""
    with ThreadPoolExecutor(max_workers=workers) as ex:
        return list(ex.map(fn, items))


def chat_json(model, messages, schema, name="out", max_tokens=1500):
    """One structured-output call. Returns the parsed dict, or None if anything went wrong."""
    try:
        r = client.chat.completions.create(
            model=model, messages=messages, temperature=0.3, max_tokens=max_tokens, extra_body=NO_THINK,
            response_format={"type": "json_schema", "json_schema": {"name": name, "schema": schema}})
        return json.loads(strip_think(r.choices[0].message.content))
    except Exception as e:
        print(f"  ({model}: {type(e).__name__}: {str(e)[:120]})")
        return None

## Part A - Guided practice: audit and clean the ticket data

### A1. Data quality dimensions

The chapter's "Data Quality" section lists what makes training data good: it is **relevant** to the task, **aligned with task requirements**, **consistent**, **correctly formatted**, **sufficiently unique**, and **compliant** with policies. Real PSA data fails in boring, predictable ways. Start with cheap **profiling** (quick counts that show what's in the data and what's wrong with it) before spending any LLM calls.

**TODO (A1a):** finish `profile_quality(df)`. Count closed tickets with no resolution note, closed tickets whose note has fewer than 8 words (like "fixed" or "Done."), and notes that say the ticket is a duplicate.

In [ ]:
def profile_quality(df):
    closed = df[df["status"] == "closed"]
    words = closed["resolution_notes"].fillna("").str.split().str.len()
    return {
        "closed_tickets": len(closed),
        "null_notes": 0,          # TODO: closed tickets with no resolution_notes at all (pandas .isna() helps)
        "short_notes": 0,         # TODO: closed tickets whose note has 1-7 words (use the words Series above)
        "notes_say_duplicate": 0, # TODO: notes that mention "duplicate" in any capitalization (.str.contains has case= and na= options)
        "rmm_alerts_without_contact": int((df["channel"].eq("rmm_alert") & df["contact_id"].isna()).sum()),
    }

prof = profile_quality(tickets)
email_case = int((contacts["email"] != contacts["email"].str.lower()).sum())
prof["contact_emails_not_lowercase"] = email_case
prof

**Reading the output:** `closed_tickets` is how many tickets are closed. `null_notes` counts closed tickets with no resolution note at all (null means "no value"), `short_notes` those whose note is 1-7 words, and `notes_say_duplicate` the notes that contain the word "duplicate" (these three read 0 until you finish the TODO). `rmm_alerts_without_contact` counts tickets opened automatically by Larkspur's monitoring software (RMM, remote monitoring and management) rather than by a person, so they have no requester. `contact_emails_not_lowercase` counts contact email addresses with capital letters.

After you finish the TODO, look at a few of the "duplicate" notes: show the `ticket_id` and `resolution_notes` columns of the tickets whose note mentions "duplicate" (reuse the condition you wrote for `notes_say_duplicate`). Are they all duplicate *tickets*? A keyword search is a weak signal; keep that in mind for A2.

**TODO (A1b):** map each issue to the chapter's quality dimension it mainly violates. Use one of: `relevant`, `aligned`, `consistent`, `correctly_formatted`, `sufficiently_unique`, `compliant`. Each dimension is defined in the chapter's "Data Quality" section; read those definitions before you choose. The definitions overlap, so more than one answer can be defended for some issues: pick the one you think fits best. The check only confirms you used valid names; Q2 grades your reasoning.

Worked example: duplicate tickets mainly violate **sufficiently unique**, because the same example appearing twice makes a model over-learn it. That one is filled in for you; do the other three the same way.

In [ ]:
DIMENSIONS = {"relevant", "aligned", "consistent", "correctly_formatted", "sufficiently_unique", "compliant"}
ISSUE_DIMENSION = {
    "duplicate_tickets": "sufficiently_unique",   # worked example (see above)
    "wrong_category_labels": "",      # TODO
    "short_or_missing_notes": "",     # TODO (would a note like "fixed" teach a model the closure standard?)
    "email_capitalization": "",       # TODO
}

In [ ]:
ck.check("A1_profile_counts", 0 < prof["null_notes"] < 50 and 0 < prof["short_notes"] < 200 and prof["notes_say_duplicate"] > 0, 2,
         "profile_quality should find some null notes, some 1-7 word notes, and some notes mentioning 'duplicate'.")
ck.check("A1_dimensions", all(v in DIMENSIONS for v in ISSUE_DIMENSION.values()), 1,
         "Fill every ISSUE_DIMENSION value with one of the six dimension names.")
for k in ("null_notes", "short_notes", "notes_say_duplicate", "contact_emails_not_lowercase"):
    ck.record(k, int(prof[k]))

### A2. Near-duplicate tickets with embeddings

The chapter's "Deduplicate Data" section asks what counts as a duplicate (must texts match exactly, or is an 80% overlap enough?) and lists ways to find them: pairwise comparison with exact, n-gram, fuzzy, or semantic similarity; hashing (MinHash, Bloom filters); and dimensionality reduction. (An n-gram is a run of n words in a row; fuzzy matching allows small differences such as typos; **semantic** similarity compares meaning rather than exact words. You'll use semantic similarity, so you don't need the other techniques here.) Duplicate *submissions* here are the same requester sending the same problem twice (often by a different channel, minutes apart), so the text is similar but not identical. Exact matching misses them; pairwise semantic similarity with embeddings catches them.

We **embed** every ticket's subject + description with the `embed` model, turning each one into an embedding: a list of numbers that captures its meaning (as in Lab 08). That takes 24 requests of 64 tickets each. Then we compare tickets **from the same client created close together in time**. That filter matters: comparing all 1.1 million pairs is the expensive part the chapter warns about, and it would flag every "password reset" as a duplicate of every other one.

In [ ]:
texts = (tickets["subject"].fillna("") + "\n" + tickets["description"].fillna("").str[:1000]).tolist()

def embed(texts, batch=64):
    vecs = []
    for i in range(0, len(texts), batch):
        r = client.embeddings.create(model=MODELS["embed"], input=texts[i:i + batch])
        vecs += [d.embedding for d in sorted(r.data, key=lambda d: d.index)]
        time.sleep(0.5)
    E = np.array(vecs, dtype=np.float32)
    return E / (np.linalg.norm(E, axis=1, keepdims=True) + 1e-9)

E = embed(texts)
print("embeddings:", E.shape)

**Reading the output:** `E.shape` is (number of tickets, numbers in each embedding): one embedding per ticket.

**TODO (A2):** finish `find_near_duplicates`. For each candidate pair `(i, j)` (same client, `j` created within `window_hours` after `i`), the code already computes `sim`, the cosine similarity of the two tickets. (Cosine similarity scores how alike two embeddings are: 1.0 means the same meaning, values near 0 mean unrelated. `embed()` scales every embedding to length 1, so a dot product of two rows gives it directly.) Your part is the decision: keep the pair if `sim` is at least `threshold`, and return a list of dicts with `original`, `duplicate` (the later ticket ID), and `sim`.

In [ ]:
def find_near_duplicates(df, E, threshold=0.90, window_hours=6):
    df = df.reset_index(drop=True)
    pairs = []
    for _, grp in df.sort_values("created").groupby("client_code"):
        idx = grp.index.tolist()
        for a, i in enumerate(idx):
            for j in idx[a + 1:]:
                if (df.at[j, "created"] - df.at[i, "created"]).total_seconds() > window_hours * 3600:
                    break
                sim = float(E[i] @ E[j])   # cosine similarity (given: the embeddings are normalized)
                # TODO: if sim reaches threshold, append a dict with the two ticket_ids (keys original and duplicate)
                #       and sim rounded to 3 places.
                pass
    return pairs

In [ ]:
for th in (0.80, 0.85, 0.90, 0.95):
    print(f"threshold {th}: {len(find_near_duplicates(tickets, E, th))} pairs")

DUP_THRESHOLD = 0.85          # change after looking at the sweep and the examples below
dups = pd.DataFrame(find_near_duplicates(tickets, E, DUP_THRESHOLD), columns=["original", "duplicate", "sim"])
by_id = tickets.set_index("ticket_id")

def show_pair(r):
    a, b = by_id.loc[r["original"]], by_id.loc[r["duplicate"]]
    print(f"{r['sim']:.3f}  {r['original']} ({a['channel']}, {a['contact_id']}) vs {r['duplicate']} ({b['channel']}, {b['contact_id']})")
    print("   ", a["subject"], "|", b["subject"])

# pairs just ABOVE your threshold (flagged) and just BELOW it (not flagged): where is the line?
near = pd.DataFrame(find_near_duplicates(tickets, E, DUP_THRESHOLD - 0.05), columns=["original", "duplicate", "sim"])
print(f"--- lowest-scoring pairs that ARE flagged (sim >= {DUP_THRESHOLD}):")
for _, r in dups.sort_values("sim").head(3).iterrows():
    show_pair(r)
print(f"--- highest-scoring pairs that are NOT flagged ({DUP_THRESHOLD - 0.05:.2f} <= sim < {DUP_THRESHOLD}):")
for _, r in near[near["sim"] < DUP_THRESHOLD].sort_values("sim", ascending=False).head(3).iterrows():
    show_pair(r)

same_contact = sum(by_id.loc[r["original"], "contact_id"] == by_id.loc[r["duplicate"], "contact_id"] for _, r in dups.iterrows())
note_says_dup = set(tickets.loc[tickets["resolution_notes"].str.contains(r"duplicate of|as duplicate|duplicate submission", case=False, na=False), "ticket_id"])
print(f"\nflagged pairs: {len(dups)}; same requester: {same_contact}; "
      f"flagged 'duplicate' tickets whose note also says duplicate: {len(set(dups['duplicate']) & note_says_dup)} of {len(note_says_dup)}")

ok_pairs = all(by_id.loc[r["original"], "client_code"] == by_id.loc[r["duplicate"], "client_code"] for _, r in dups.iterrows())
ck.check("A2_embeddings", E.shape[0] == len(tickets), 1, "Embed every ticket (one vector per row).")
ck.check("A2_pairs_valid", len(dups) > 0 and ok_pairs and (dups["sim"] >= DUP_THRESHOLD).all(), 2,
         "find_near_duplicates should return same-client pairs with sim >= threshold. Did you append the dict?")
ck.check("A2_plausible_count", 5 <= len(dups) <= 80, 1,
         "Expect roughly 10-40 pairs at a sensible threshold. Very many = threshold too low; zero = check the TODO.")
ck.record("dup_threshold", DUP_THRESHOLD)
ck.record("n_dup_pairs", int(len(dups)))
ck.record("dup_flagged_ids", sorted(set(dups["duplicate"])))

**Reading the output:** first, how many pairs each similarity threshold would flag (a higher threshold is stricter, so it flags fewer pairs). Then the pairs closest to your threshold on each side: the similarity score, the two ticket IDs, each with its channel (how the ticket came in) and the requester's contact ID, and both subjects on the next line.

**Choosing the threshold:** a duplicate *submission* is the same person reporting the same problem twice: same contact ID, similar subject, usually minutes apart. Two *different* people reporting the same outage are not duplicates (both tickets are real). If the flagged pairs just above the line look like the second kind, raise `DUP_THRESHOLD`; if the unflagged pairs just below it look like the first kind, lower it. Then re-run this cell. The last line counts the flagged pairs, how many of them come from the same requester, and how many of the later tickets have a resolution note that also calls them a duplicate.

### A3. Suspicious labels: nearest neighbors, then an LLM relabeling pass

About 1,500 tickets is too many to relabel with an LLM on a student rate limit, and most labels are fine. So we use a cheap filter first and the expensive check only on what it flags. (The chapter's "Data verification" section suggests anomaly detection to find outlier examples, and AI verifiers to score examples; this is a simple version of both.)

1. **Neighbor vote (free).** For each ticket, look at the categories of its 10 most similar tickets (its **nearest neighbors**, by cosine similarity of the embeddings from A2). If most neighbors agree on a *different* category, the label is suspicious.
2. **LLM relabel (about 12 requests).** Ask `qwen-moe` to label the suspicious tickets, a few priority-suspicious ones, and a random control group, using the categories and Larkspur's priority definitions. The **control group** is 15 tickets picked at random, not because anything looked wrong. It tells you how often the LLM disagrees with labels that are probably fine: that's the LLM's own noise.

**Why labels matter for the dataset you build:** every ticket example in B5 gives the model the ticket's category and priority as part of its input (look at `ticket_prompt()` in B5). A wrong label teaches the model a wrong link, for example that a printer note belongs to a "network" P1 ticket. That's why A4 drops tickets whose category is in doubt.

**What the next cell does, in plain words** (you don't edit it, but you should be able to follow it):
- `sims = E @ E.T` scores every ticket against every other ticket (cosine similarity, as in A2); `fill_diagonal(..., -1)` stops a ticket from counting as its own neighbor.
- `np.argsort(-sims, axis=1)[:, :10]` sorts each ticket's scores from highest to lowest and keeps the 10 most similar tickets: its nearest neighbors.
- `Counter(...).most_common(1)` finds the most common category among those 10 neighbors and how many of them have it. A ticket is flagged when at least 6 of 10 neighbors share a different category.
- The `outage`, `single`, and `asks` patterns are regular expressions: each asks "does the text contain any of these words?" They flag priorities that don't match the text, using Larkspur's priority definitions (shown in the comment).
- Finally it builds the sample to relabel: up to 30 category-flagged tickets, up to 15 priority-flagged ones, and 15 random controls.

In [ ]:
CATS = sorted(tickets["category"].unique())
sims = E @ E.T
np.fill_diagonal(sims, -1)
nn = np.argsort(-sims, axis=1)[:, :10]
cats = tickets["category"].to_numpy()
votes = [Counter(cats[row]).most_common(1)[0] for row in nn]
tickets["nn_category"] = [c for c, _ in votes]
tickets["nn_share"] = [n / 10 for _, n in votes]
knn_flag = tickets[(tickets["nn_category"] != tickets["category"]) & (tickets["nn_share"] >= 0.6)]
print(len(knn_flag), "tickets where >= 6 of 10 neighbors have another category")

# Priority cues from the definitions: P1 = business down (so a P1 written in the first person
# about one user's problem is suspicious), P4 = requests (so a P4 machine alert is suspicious),
# and a P2 that reads like a request or a "how do I" question is suspicious.
text = tickets["subject"].fillna("") + " " + tickets["description"].fillna("")
outage = r"\b(?:down|outage|nobody|no one|everyone|everybody|all users|whole office|can't work|cannot work|site)\b"
single = r"\b(?:i|my|me|i'm|i've)\b"
asks = r"\b(?:how do i|can i|is there a way|request|order|quote|new hire|starts)\b"
human = tickets["channel"].ne("rmm_alert")
prio_flag = tickets[(tickets["priority"].eq("P1") & human & text.str.contains(single, case=False) & ~text.str.contains(outage, case=False))
                    | (tickets["priority"].eq("P4") & ~human)
                    | (tickets["priority"].eq("P2") & human & text.str.contains(asks, case=False))]
print(len(prio_flag), "tickets where the text and the priority look mismatched")

sample_ids = list(knn_flag.sort_values("nn_share", ascending=False)["ticket_id"][:30])
sample_ids += [t for t in prio_flag["ticket_id"] if t not in sample_ids][:15]
controls = tickets[~tickets["ticket_id"].isin(sample_ids)].sample(15, random_state=632)["ticket_id"].tolist()
sample = tickets.set_index("ticket_id").loc[sample_ids + controls].reset_index()
sample["group"] = ["flagged"] * len(sample_ids) + ["control"] * len(controls)
print(len(sample), "tickets to relabel:", Counter(sample["group"]))

In [ ]:
PRIORITY_RULES = """P1 Critical: business down or most users unable to work (internet outage, server down, email down company-wide).
P2 High: multiple users affected or a key function degraded; or a single VIP/executive fully blocked.
P3 Normal: a single user affected, a workaround may exist.
P4 Low: requests, questions, scheduled changes, new user setups."""

LABEL_SCHEMA = {"type": "object", "properties": {"labels": {"type": "array", "items": {
    "type": "object", "properties": {
        "ticket_id": {"type": "string"},
        "category": {"type": "string", "enum": CATS},
        "priority": {"type": "string", "enum": ["P1", "P2", "P3", "P4"]}},
    "required": ["ticket_id", "category", "priority"]}}}, "required": ["labels"]}


def relabel_batch(batch):
    items = "\n\n".join(f"[{r.ticket_id}] Subject: {r.subject}\n{str(r.description)[:600]}\nResolution: {str(r.resolution_notes)[:300]}"
                        for r in batch.itertuples())
    msgs = [{"role": "system", "content": "You label IT help-desk tickets for a managed service provider."},
            {"role": "user", "content": f"Categories: {', '.join(CATS)}\n\nPriority definitions:\n{PRIORITY_RULES}\n\n"
                                        f"Label each ticket below with the best category and priority.\n\n{items}"}]
    out = chat_json(GEN_MODEL, msgs, LABEL_SCHEMA, "labels")
    return (out or {}).get("labels", [])

batches = [sample.iloc[i:i + 5] for i in range(0, len(sample), 5)]
llm_labels = [lab for labs in parallel_map(relabel_batch, batches) for lab in labs]
llm = pd.DataFrame(llm_labels, columns=["ticket_id", "category", "priority"]).drop_duplicates("ticket_id")
llm = llm[llm["ticket_id"].isin(sample["ticket_id"])]
rel = sample.merge(llm, on="ticket_id", how="inner", suffixes=("", "_llm"))
print(len(rel), "tickets relabeled by the LLM")

**TODO (A3):** add two boolean columns to `rel`: `cat_disagree` (the LLM's `category_llm` differs from `category`) and `pri_disagree` (`priority_llm` differs from `priority`). Then the cell prints disagreement rates for the flagged group vs the control group. (A boolean column holds only `True` or `False`; comparing two columns with `!=` gives you one.)

In [ ]:
rel["cat_disagree"] = False   # TODO
rel["pri_disagree"] = False   # TODO

In [ ]:
if len(rel):
    print(rel.groupby("group")[["cat_disagree", "pri_disagree"]].mean().round(2))
    show = rel[rel["cat_disagree"] | rel["pri_disagree"]]
    display(show[["ticket_id", "group", "subject", "category", "category_llm", "priority", "priority_llm"]].head(15))

ck.check("A3_knn_flags", 0 < len(knn_flag) < 400, 1, "The neighbor vote should flag some (not hundreds of) tickets.")
ck.check("A3_llm_relabeled", len(rel) >= 30 and rel["category_llm"].isin(CATS).all() if len(rel) else False, 2,
         "At least 30 tickets should come back with valid LLM labels. Check the error messages printed by chat_json.")
ck.check("A3_disagreement_columns", len(rel) > 0 and rel["cat_disagree"].dtype == bool
         and bool((rel["cat_disagree"] == (rel["category_llm"] != rel["category"])).all()), 1,
         "cat_disagree should be True exactly where the LLM's category differs from the original category column.")
susp_cat = sorted(rel.loc[rel["cat_disagree"] & rel["group"].eq("flagged"), "ticket_id"])
susp_pri = sorted(rel.loc[rel["pri_disagree"] & rel["group"].eq("flagged"), "ticket_id"])
ck.record("n_knn_flagged", int(len(knn_flag)))
ck.record("n_relabeled", int(len(rel)))
ck.record("n_category_disagree_flagged", len(susp_cat))
ck.record("n_priority_disagree_flagged", len(susp_pri))
ck.record("control_category_disagree_rate", float(rel.loc[rel["group"].eq("control"), "cat_disagree"].mean()) if len(rel) else None)
ck.record("suspect_category_ids", susp_cat)
ck.record("suspect_priority_ids", susp_pri)

**Reading the output:** the small table shows, for each group, the fraction of relabeled tickets where the LLM's label differs from the original (0.25 means 1 in 4), for category and for priority. The larger table lists tickets with at least one disagreement: the original labels (`category`, `priority`) next to the LLM's (`category_llm`, `priority_llm`). A disagreement is a reason to look, not proof that the original label is wrong.

### A4. Clean

Cleaning is a set of explicit, documented decisions (the chapter's "Clean and Filter Data"). We won't "fix" labels by trusting the LLM blindly; for the SFT dataset we simply **exclude** tickets we have doubts about. Cheap, and it doesn't add new errors. (We drop only disputed tickets from the **flagged** group. The control tickets were picked at random to measure how noisy the LLM itself is, not to be cleaned.)

**TODO (A4):** build `clean` from `tickets` by dropping (1) the later ticket of each duplicate pair, (2) tickets whose category the LLM disputed (`susp_cat`), and (3) set a `note_ok` column: note present and at least 8 words. Also lower-case contact emails.

In [ ]:
clean = tickets.copy()
# TODO: drop the near-duplicate tickets you found in A2 (the "duplicate" column of dups).
# TODO: drop the tickets listed in susp_cat.
clean["note_ok"] = False   # TODO: True when the note has at least 8 words (treat a missing note as empty)

contacts_clean = contacts.copy()
# TODO: make every contact email address lowercase.

print(f"{len(tickets)} -> {len(clean)} tickets; {int(clean['note_ok'].sum())} with usable notes")

In [ ]:
ck.check("A4_dedupe_and_filter", len(clean) < len(tickets) and not clean["ticket_id"].isin(set(dups["duplicate"])).any(), 2,
         "clean must drop the later ticket of every duplicate pair.")
ck.check("A4_note_ok", clean["note_ok"].dtype == bool and 900 < int(clean["note_ok"].sum()) < len(clean), 1,
         "note_ok should be True for most closed tickets and False for missing/short notes.")
ck.check("A4_emails_normalized", (contacts_clean["email"] == contacts_clean["email"].str.lower()).all(), 1,
         "Lower-case the contact emails (email matching should be case-insensitive).")
ck.record("n_clean_tickets", int(len(clean)))

## Part B - Applied lab: build the SFT dataset for Lab 11

### B1. Generate only from CURRENT documents

Synthetic data inherits every flaw of its source. The corpus contains **outdated** documents next to current ones (old runbook versions, older editions of policies and client handouts). If you generate training pairs from an outdated page, you are teaching the model the wrong answer, and you will not be able to tell which examples are wrong later. So we decide what is current **before** generating:

Every corpus document starts with a small block of metadata (its front matter) that includes `doc_id`, `version`, and `effective_date` (the date that version took effect).

1. For each `doc_id`, keep only the version with the newest `effective_date` (old runbook versions share a `doc_id`).
2. For documents that are **not runbooks**, drop anything effective **before 2025-01-01**. Larkspur changed prices, business hours, MFA and escalation procedures in 2025-2026; a non-runbook page older than that predates those changes. (Runbooks carry explicit versions and are reviewed yearly, so a 2024 runbook that is the newest version of its `doc_id` is still current.)

Run the next cell first: it loads the documents and shows the fields of one of them (everything except the text).

**TODO (B1):** write `is_current(doc, docs)` for those two rules. Hints: a document's type is in `doc["doc_type"]` (runbooks have `"runbook"`). Dates are strings like `"2025-06-02"`; comparing them with `>=` or `max()` works because they are written year-month-day, so text order is the same as date order.

In [ ]:
docs = load_docs()
print(len(docs), "documents. Fields of the first one:")
print({k: v for k, v in docs[0].items() if k != "text"})

In [ ]:
def is_current(doc, docs):
    """True if doc is the newest version of its doc_id and (a runbook, or effective on/after 2025-01-01)."""
    # TODO: find the newest effective_date among the docs that share this doc's doc_id.
    # TODO: keep the doc only if it is that newest version AND it is a runbook or took effect on or after 2025-01-01.
    return True

current = [d for d in docs if is_current(d, docs)]
excluded = sorted(f"{d['doc_id']} v{d['version']} ({d['effective_date']})" for d in docs if d not in current)
print(len(docs), "documents ->", len(current), "current. Excluded:", *excluded, sep="\n  ")

In [ ]:
newest = {}
for d in docs:
    newest[d["doc_id"]] = max(newest.get(d["doc_id"], ""), d["effective_date"])
rule_ok = all(d["effective_date"] == newest[d["doc_id"]] and (d["doc_type"] == "runbook" or d["effective_date"] >= "2025-01-01")
              for d in current)
ck.check("B1_current_rule", rule_ok and len(current) < len(docs), 2,
         "Every kept document must be the newest version of its doc_id, and non-runbooks must be effective 2025-01-01 or later.")
ck.record("n_current_docs", len(current))

### B2. Generate grounded question/answer pairs

We split current documents into sections, pick about 50 sections (at most 2 per document, skipping revision-history tables), and ask `qwen-moe` for 5 pairs per section: about 250 candidates from 50 requests. The chapter's "Limitations to AI-generated data" warns that generated data can be low quality ("Quality control") and can imitate the right style while getting facts wrong ("Superficial imitation"), so every pair must carry **evidence**: a phrase copied from the source section. A pair is **grounded** when its evidence really appears in the source. We check that evidence ourselves; we don't take the model's word for it.

In [ ]:
def sections(doc, max_chars=2200):
    parts = re.split(r"\n(?=## )", doc["text"])
    out = []
    for n, p in enumerate(parts):
        p = p.strip()
        if len(p) < 300 or p.lower().startswith("## revision history"):
            continue
        out.append({"chunk_id": f"{doc['doc_id']}#s{n}", "doc_id": doc["doc_id"], "version": str(doc["version"]),
                    "effective_date": str(doc["effective_date"]), "title": doc["title"], "audience": doc.get("audience", ""),
                    "text": p[:max_chars]})
    return out

all_chunks = [c for d in current for c in sections(d)]
rng = random.Random(632)
by_doc = {}
for c in all_chunks:
    by_doc.setdefault(c["doc_id"], []).append(c)
N_CHUNKS = 50
chunks = []
for doc_id in sorted(by_doc):
    chunks += rng.sample(by_doc[doc_id], min(2, len(by_doc[doc_id])))
chunks = rng.sample(chunks, min(N_CHUNKS, len(chunks)))
print(len(all_chunks), "sections in current docs;", len(chunks), "chosen from", len({c['doc_id'] for c in chunks}), "documents")

QA_SCHEMA = {"type": "object", "properties": {"pairs": {"type": "array", "items": {
    "type": "object", "properties": {"question": {"type": "string"}, "answer": {"type": "string"}, "evidence": {"type": "string"}},
    "required": ["question", "answer", "evidence"]}}}, "required": ["pairs"]}

GEN_PROMPT = """Document: {doc_id} (version {version}, effective {effective_date}) - {title}. Audience: {audience}.
---
{text}
---
Write 5 question-answer pairs that a Larkspur Ridge IT technician or client might really ask, answerable ONLY from the text above.
- Each question must make sense on its own, without seeing the text (name the topic, tier, product or procedure).
- Answers: 1-3 sentences, specific (numbers, names, steps), nothing that is not in the text.
- evidence: copy, word for word, the sentence or table row from the text that supports the answer.
- Never say "the text", "the document" or "according to"."""


def generate(chunk):
    out = chat_json(GEN_MODEL, [{"role": "system", "content": "You write training data for a company's IT help-desk assistant."},
                                {"role": "user", "content": GEN_PROMPT.format(**chunk)}], QA_SCHEMA, "qa_pairs")
    return [{**p, "chunk_id": chunk["chunk_id"]} for p in (out or {}).get("pairs", []) if isinstance(p, dict)]

t0 = time.time()
candidates = [p for ps in parallel_map(generate, chunks) for p in ps]
print(f"{len(candidates)} candidate pairs in {time.time() - t0:.0f}s")
CHUNK = {c["chunk_id"]: c for c in chunks}
for p in candidates[:3]:
    print(json.dumps(p, indent=1)[:500])

**TODO (B2):** write `grounded(evidence, source_text)`. **Reading the output above:** each candidate is a JSON object with the generated `question` and `answer`, the `evidence` phrase the model says it copied, and `chunk_id`: which document and section it came from (`doc_id#s` plus the section number).

Normalize both with `normalize()` (given: it lower-cases the text and replaces Markdown symbols and punctuation with spaces, so formatting differences don't matter) and return `True` if the normalized evidence is at least 15 characters long **and** appears inside the normalized source. (Optional: also accept a fuzzy match when 85% of the evidence matches one stretch of the source and every number in the evidence appears in the source. Models often change a word or two when they "copy".)

In [ ]:
def normalize(s):
    s = re.sub(r"[*`|#>_.,;:!?()\"']", " ", str(s).lower())
    return re.sub(r"\s+", " ", s).strip()


def grounded(evidence, source_text):
    ev, src = normalize(evidence), normalize(source_text)
    # TODO: the evidence passes when it is at least 15 characters long and appears word for word in the source.
    return False


print(grounded("Server management: $175 per server per month", "## Add-ons\n| Server management | **$175 per server per month** |"))  # True: normalize() strips markdown and punctuation
print(grounded("Payment terms: Net 15.", "- Payment terms: **Net 15**."))  # True

In [ ]:
for p in candidates:
    p["grounded"] = grounded(p.get("evidence", ""), CHUNK[p["chunk_id"]]["text"])
print(f"grounded: {sum(p['grounded'] for p in candidates)} of {len(candidates)}")

ck.check("B2_candidates", len(candidates) >= 100, 2,
         "Expect about 250 candidates (5 per section). If far fewer, read the errors printed by chat_json.")
ck.check("B2_grounded_function", grounded("Payment terms: Net 15.", "- Payment terms: **Net 15**.")
         and not grounded("Payment terms: Net 30.", "- Payment terms: **Net 15**.") and not grounded("Net 15", "Net 15"), 2,
         "grounded() must accept evidence found in the source, reject evidence that is not, and reject very short evidence.")
ck.record("n_candidates", len(candidates))

### B3. Filter: AI judge plus heuristics

Two kinds of filters, as in the chapter's "Data verification" discussion:
- **AI judge** (`gemma-moe`, a different model family than the generator, so it doesn't just like its own style): scores each pair's **faithfulness** to the source section from 1 to 5 (does the answer say only what the source says?), and whether the question stands on its own (makes sense to someone who hasn't seen the document). One judge call per section (all 5 pairs at once): about 50 requests.
- **Heuristics** (simple rule-of-thumb checks, free because they need no model calls): grounded evidence, sensible lengths, no "according to the document" phrasing, and near-duplicate questions removed with embeddings.

In [ ]:
JUDGE_SCHEMA = {"type": "object", "properties": {"scores": {"type": "array", "items": {
    "type": "object", "properties": {"id": {"type": "integer"}, "faithful": {"type": "integer", "minimum": 1, "maximum": 5},
                                     "standalone": {"type": "boolean"}},
    "required": ["id", "faithful", "standalone"]}}}, "required": ["scores"]}

JUDGE_PROMPT = """Source section from {doc_id} ({title}):
---
{text}
---
Rate each question-answer pair below.
faithful: 5 = every claim in the answer is supported by the source; 3 = partly supported or vague; 1 = contradicts or is not in the source.
standalone: true if the question makes sense to someone who has not seen the source.

{pairs}"""


def judge(chunk_id):
    pairs = [p for p in candidates if p["chunk_id"] == chunk_id]
    listing = "\n".join(f"[{i}] Q: {p['question']}\n    A: {p['answer']}" for i, p in enumerate(pairs))
    out = None
    for model in (JUDGE_MODEL, GEN_MODEL):            # fall back to qwen-moe if the judge model errors
        out = chat_json(model, [{"role": "user", "content": JUDGE_PROMPT.format(pairs=listing, **CHUNK[chunk_id])}],
                        JUDGE_SCHEMA, "scores", max_tokens=600)
        if out:
            break
    for s in (out or {}).get("scores", []):
        if isinstance(s.get("id"), int) and 0 <= s["id"] < len(pairs):
            pairs[s["id"]]["faithful"] = s.get("faithful")
            pairs[s["id"]]["standalone"] = s.get("standalone")
            pairs[s["id"]]["judge"] = model
    return len(pairs)

parallel_map(judge, sorted({p["chunk_id"] for p in candidates}))
judged = [p for p in candidates if p.get("faithful") is not None]
print(f"judged {len(judged)} of {len(candidates)}; faithfulness scores:", Counter(p["faithful"] for p in judged))

**TODO (B3):** write `keep(p)`, returning `True` only if: the pair is `grounded`, the judge gave `faithful >= 4` and `standalone` is true, the question is 15-300 characters, the answer 20-800 characters, and neither matches `META` (a regular expression, a text pattern, that spots phrases like "according to the document"). The funnel table shows how many pairs each rule removes.

In [ ]:
META = re.compile(r"\b(the (text|document|passage|excerpt|section|source)|according to)\b", re.I)

def keep(p):
    q, a = p.get("question", ""), p.get("answer", "")
    # TODO: combine the rules described above into one boolean
    return True

In [ ]:
funnel = {"candidates": len(candidates),
          "grounded": sum(bool(p.get("grounded")) for p in candidates),
          "judge faithful>=4": sum(bool(p.get("grounded")) and (p.get("faithful") or 0) >= 4 for p in candidates),
          "all keep() rules": sum(keep(p) for p in candidates)}
kept = [p for p in candidates if keep(p)]

# near-duplicate questions (same question asked twice in different words)
if kept:
    Q = embed([p["question"] for p in kept])
    drop = set()
    for i in range(len(kept)):
        if i not in drop:
            for j in np.where(Q[i + 1:] @ Q[i] >= 0.92)[0] + i + 1:
                drop.add(int(j))
    kept = [p for i, p in enumerate(kept) if i not in drop]
funnel["after question dedupe"] = len(kept)
display(pd.Series(funnel, name="pairs"))
print("judge model used:", dict(Counter(p.get("judge", "none") for p in judged)))
if len(kept) < 110:
    print(f"\nWARNING: only {len(kept)} pairs survived. With the 120 ticket notes in B4 you need about 100+ to reach "
          "200 training examples in B5. Raise N_CHUNKS in B2 (try 70) and re-run from B2 down to here now.")

rejected = [p for p in candidates if not keep(p)]
for p in rejected[:4]:
    print(f"\nREJECTED (grounded={p.get('grounded')}, faithful={p.get('faithful')}, standalone={p.get('standalone')})\n Q: {p.get('question')}\n A: {p.get('answer')}")

ck.check("B3_judge_ran", len(judged) >= 0.8 * max(1, len(candidates)), 2, "The judge should score at least 80% of candidates.")
ck.check("B3_filter_applied", funnel["all keep() rules"] < funnel["candidates"] and 0 < funnel["after question dedupe"], 1,
         "keep() should remove some pairs but not all of them.")
ck.record("qa_funnel", funnel)

**Reading the funnel:** each row is how many pairs are left after one more rule, from all candidates down to the final set (`after question dedupe` also removes near-duplicate questions using embeddings). The biggest drop tells you which rule removes the most. Below it are a few rejected pairs with the values that decided their fate: `grounded` (evidence found in the source), `faithful` (judge score 1-5, `None` if not judged) and `standalone`.

The `judge model used` line shows which model scored the pairs. If the judge (`gemma-moe`) failed for some sections, `judge()` falls back to `qwen-moe`, the same model that generated them; say so in Q4 if that happened. If you see a WARNING about too few pairs, fix it now rather than at B5.

### B4. Real "ticket -> resolution note" pairs

Synthetic Q&A teaches facts from documents. Real resolution notes teach Larkspur's **form**: how a good closure note is written (Larkspur's standard is in runbook RB-GEN-004: open `corpus/larkspur-corpus/runbooks/RB-GEN-004_v3.md` if you want to see it). We take them from the **cleaned** tickets, but with one more lineage decision: technicians' notes follow the procedure **in effect when the ticket was worked**. The data dictionary ("Events and patterns") says the identity-verification, MFA and after-hours procedures changed during the first half of 2025, the last one on **2025-06-02**. Notes older than that can teach a retired procedure.

**TODO (B4):** filter `clean` to tickets that are closed, `note_ok`, have notes of at least 15 words, and were created on or after `CUTOFF`. Combine conditions with `&` and wrap each one in parentheses, like `(cond1) & (cond2)`; without them pandas raises a confusing error.

In [ ]:
CUTOFF = "2025-06-02"
words = clean["resolution_notes"].fillna("").str.split().str.len()
pool = clean[clean["status"] == "closed"]   # TODO: also require note_ok, a note of at least 15 words (the words Series above), and creation on or after CUTOFF
print(len(pool), "tickets in the pool")

In [ ]:
N_TICKETS = 140   # 120 train + 20 test
per_cat = max(1, N_TICKETS // pool["category"].nunique()) if len(pool) else 1
parts = [g.sample(min(len(g), per_cat), random_state=632) for _, g in pool.groupby("category")]
picked = pd.concat(parts) if parts else pool
if len(picked) < N_TICKETS and len(pool):
    picked = pd.concat([picked, pool.drop(picked.index).sample(min(len(pool) - len(picked), N_TICKETS - len(picked)), random_state=632)])
print(len(picked), "ticket pairs; categories:", dict(Counter(picked["category"])))

ck.check("B4_ticket_pool_rules", len(pool) > 100 and (pool["created_at"] >= CUTOFF).all() and pool["note_ok"].all()
         and (pool["resolution_notes"].str.split().str.len() >= 15).all(), 2,
         "The pool must only contain closed tickets created on/after CUTOFF with usable notes of 15+ words.")
ck.record("n_ticket_pool", int(len(pool)))

### B5. Format, split, record lineage, save

Lab 11 trains with TRL's `SFTTrainer` (TRL is Hugging Face's library for training language models), which reads **chat format**: one JSON object per line (a **JSONL** file) with a `messages` list. We also keep a `meta` object on every line (lineage: which document and version, or which ticket, it came from; which models generated and judged it). Lab 11 drops `meta` before training, but you keep it: if a document changes or a source turns out to be wrong, lineage tells you exactly which examples to remove (the chapter's warning about "obscure data lineage").

The test split (40 examples: 20 Q&A + 20 tickets) is held out; we also remove training questions that are near-duplicates of test questions, so Lab 11 doesn't measure memorization of the test set. (Test examples leaking into training is called **data leakage**.)

In [ ]:
SYSTEM = "You are the Larkspur Ridge IT help-desk assistant. Answer accurately and concisely using Larkspur's current policies and procedures."

def qa_example(p):
    c = CHUNK[p["chunk_id"]]
    return {"messages": [{"role": "system", "content": SYSTEM}, {"role": "user", "content": p["question"]},
                         {"role": "assistant", "content": p["answer"]}],
            "meta": {"source_type": "doc_qa", "source_id": c["doc_id"], "version": c["version"],
                     "effective_date": c["effective_date"], "chunk_id": c["chunk_id"], "generator": GEN_MODEL,
                     "judge": str(p.get("judge", "")), "judge_score": str(p.get("faithful", "")), "split": ""}}

def ticket_prompt(r):
    return (f"Write the Harbor PSA resolution note for this ticket.\nClient tier: {r['tier']}\n"
            f"Category: {r['category']} / {r['subcategory']}\nPriority: {r['priority']}\n"
            f"Subject: {r['subject']}\nDescription: {str(r['description'])[:800]}")

def ticket_example(r):
    return {"messages": [{"role": "system", "content": SYSTEM}, {"role": "user", "content": ticket_prompt(r)},
                         {"role": "assistant", "content": r["resolution_notes"]}],
            "meta": {"source_type": "ticket_note", "source_id": r["ticket_id"], "version": "",
                     "effective_date": r["created_at"][:10], "chunk_id": "", "generator": "human (technician)",
                     "judge": "", "judge_score": "", "split": ""}}

qa = [qa_example(p) for p in kept]
tk = [ticket_example(r) for _, r in picked.iterrows()]
random.Random(632).shuffle(qa)
random.Random(633).shuffle(tk)
n_qa_test = min(20, len(qa) // 3)
test = qa[:n_qa_test] + tk[:40 - n_qa_test]
train = qa[n_qa_test:] + tk[40 - n_qa_test:]

# leakage check: drop training Q&A whose question is a near-duplicate of a test question
leaks = 0
if qa[:n_qa_test] and qa[n_qa_test:]:
    Qt = embed([e["messages"][1]["content"] for e in qa[:n_qa_test]])
    Qr = embed([e["messages"][1]["content"] for e in qa[n_qa_test:]])
    leak_idx = set(np.where((Qr @ Qt.T).max(axis=1) >= 0.92)[0])
    leaks = len(leak_idx)
    train = [e for i, e in enumerate(qa[n_qa_test:]) if i not in leak_idx] + tk[40 - n_qa_test:]
random.Random(634).shuffle(train)
for e in train:
    e["meta"]["split"] = "train"
for e in test:
    e["meta"]["split"] = "test"

def write_jsonl(path, rows):
    with open(path, "w", encoding="utf-8") as f:
        for r in rows:
            f.write(json.dumps(r, ensure_ascii=False) + "\n")

write_jsonl(OUT / "larkspur_sft_train.jsonl", train)
write_jsonl(OUT / "larkspur_sft_test.jsonl", test)
card = pd.DataFrame([e["meta"] for e in train + test])
print(f"train {len(train)}  test {len(test)}  (removed {leaks} leaked training questions)")
display(card.groupby(["split", "source_type"]).size().rename("examples"))
print("documents covered:", card.loc[card.source_type.eq("doc_qa"), "source_id"].nunique())
print("saved to", OUT.resolve())

**Reading the output:** the first line gives the train and test sizes, and how many training questions were dropped because they nearly matched a test question. The table counts examples by split (`train` or `test`) and source type (`doc_qa` = generated from a document, `ticket_note` = taken from a real ticket). Then: how many different documents the Q&A examples cover, and where the files were saved.

In [ ]:
def valid_example(e):
    m = e.get("messages", [])
    return (len(m) == 3 and [x["role"] for x in m] == ["system", "user", "assistant"]
            and all(isinstance(x["content"], str) and x["content"].strip() for x in m)
            and e.get("meta", {}).get("source_id"))

reloaded = [json.loads(l) for l in open(OUT / "larkspur_sft_train.jsonl", encoding="utf-8")]
current_keys = {(d["doc_id"], str(d["version"])) for d in current}
ck.check("B5_train_size", 200 <= len(train) <= 400, 2,
         "Aim for 200-400 training examples. Too few? Raise N_CHUNKS or N_TICKETS (watch the request budget), or loosen one filter and justify it in Q4.")
ck.check("B5_test_size", len(test) == 40, 1, "The test split should have exactly 40 examples.")
ck.check("B5_format_and_lineage", len(reloaded) == len(train) and all(valid_example(e) for e in reloaded), 2,
         "Every line needs messages [system, user, assistant] with text, and meta.source_id.")
ck.check("B5_only_current_sources", all((e["meta"]["source_id"], e["meta"]["version"]) in current_keys
                                        for e in train + test if e["meta"]["source_type"] == "doc_qa"), 1,
         "Every doc_qa example must come from a CURRENT document version.")
ck.record("n_train", len(train))
ck.record("n_test", len(test))
ck.record("n_train_doc_qa", int(sum(e["meta"]["source_type"] == "doc_qa" for e in train)))
ck.record("n_train_ticket_note", int(sum(e["meta"]["source_type"] == "ticket_note" for e in train)))
ck.record("n_leaks_removed", int(leaks))

## Written analysis (40 points)

Use your own numbers (they are in the summary at the end) and cite Chapter 8 (and Chapter 7 for Q4) by section name or concept.

**Q1. Quality vs quantity.** Report your Q&A funnel (candidates -> grounded -> judge -> final) and your final train/test sizes. The chapter argues that a small amount of high-quality data can beat a large amount of noisy data. If you could have 600 unfiltered pairs or your filtered set, which would you train on in Lab 11, and why? Use your judge-score distribution and at least one rejected example as evidence.

*Your answer:*

**Q2. Label noise.** Report what you found in Part A: near-duplicate pairs (and your threshold), neighbor-vote flags, and the LLM disagreement rate for flagged vs control tickets. What does the control-group rate tell you about trusting the LLM as a labeler? Pick **two** of the issue types from A1b: for each, name the data quality dimension from the chapter's "Data Quality" section it violates (justify your choice) and what it would do to a model trained on this data.

*Your answer:*

**Q3. Synthetic data risks.** The chapter lists limitations of AI-generated data (for example quality control, superficial imitation, potential model collapse, and obscure data lineage). Pick two and show where each appeared, or could have appeared, in *your* pipeline, pointing at a specific candidate pair or step. Then explain why we generated only from current documents: what would have happened to the dataset if an outdated document version had been included?

*Your answer:*

**Q4. What you filtered, and why fine-tune at all.** Justify **three** of the filter and lineage decisions in your pipeline (choose from: current-docs rule, judge threshold, judge model choice, question dedupe, the 2025-06-02 ticket cutoff, the test-leakage check). Then, using the "When to Finetune" section of Chapter 7 (especially "Finetuning and RAG": information-based versus behavior-based failures): which half of your dataset (doc Q&A or ticket notes) do you expect fine-tuning to help more, and why? You will test this prediction in Lab 11; a short, reasoned guess is enough.

*Your answer:*

## AI-use reflection (10 points)

1. What did you use AI assistance for in this lab (tools, which steps)?

*Your answer:*

2. Where did the AI help most, and where was it wrong or unhelpful? How did you notice?

*Your answer:*

3. What is one thing you now understand that you could explain without AI help?

*Your answer:*

4. In this lab an AI generated your training data and another AI judged it. Where in the pipeline did you rely on your *own* judgment instead, and would you trust the pipeline without that step?

*Your answer:*

## Submit

Run this last, save, upload the .ipynb. **Also keep** `my_dataset/larkspur_sft_train.jsonl` and `my_dataset/larkspur_sft_test.jsonl`: you upload them to Colab in Lab 11 (Canvas doesn't need them).

In [ ]:
ck.report()